# H4a Wzmianki o obozach

## 1. Konfiguracja

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.config import CAMP_TERMS, POLITICAL_SUBREDDITS, PROCESSED_DIR
from src.terms import find_mentions

H4A_DIR = PROCESSED_DIR / "h4a"
H4A_DIR.mkdir(parents=True, exist_ok=True)
assert any(CAMP_TERMS[camp][term_type] for camp in CAMP_TERMS for term_type in CAMP_TERMS[camp]), "CAMP_TERMS jest pusty"

## 2. Próba subredditów politycznych

In [2]:
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "subreddit", "camp", "cycle", "week_start", "phase", "text"])
sample = sample[sample["subreddit"].isin(POLITICAL_SUBREDDITS)].reset_index(drop=True)
sample.shape

(455045, 7)

## 3. Komentarze wspominające dokładnie jeden obóz

In [3]:
mentions = find_mentions(sample["text"], CAMP_TERMS)
data = sample.join(mentions, how="inner")
data["target"] = (data["mentioned_camp"] == data["camp"]).map({True: "own", False: "out"})
data.to_parquet(H4A_DIR / "mentions.parquet", index=False)
print(len(data), "z", len(sample))

95312 z 455045


## 4. Liczebność koszyków

In [4]:
data.groupby(["subreddit", "cycle", "phase", "target"]).size().unstack("target")

target                             out   own
subreddit    cycle phase                    
Conservative 2012  bazowa          345   218
                   powyborcza      262    95
                   przedwyborcza  1476  1408
                   przejsciowa     280   180
                   szczytowa       319   246
...                                ...   ...
democrats    2024  bazowa         1010   369
                   powyborcza      305   199
                   przedwyborcza  2738  1445
                   przejsciowa     518   272
                   szczytowa       509   331

[80 rows x 2 columns]